# Rule R on the small colour sets: 384-px tiles (overlap 96) for every image, versus the whole image

Rule R was chosen on Urban100's development images (`urban100-tiles`), where 512-px tiles beat the whole image by
+0.05 / +0.09 / +0.15 dB on the held-out images and 384-px tiles did better still. Here it is applied unchanged to
every image of CBSD68 (481×321), Kodak24 (768×512) and McMaster (500×500) at σ 15 / 25 / 50. Nothing is selected on
these images (amendment 6).

Arms: whole image ×1 / ×2 (horizontal flip), 384-px tiles ×1 / ×2 (rule R), 512-px tiles ×2 (secondary).

Added after the Phase-1 extreme critique (test P1c, pre-registered in the decision log): DMID's own S_t = 2 chain at
η = 0 on the whole image. It uses three evaluations, N−1 → ⌊N/2⌋ → 0 → clean, with DMID's integer N; the earlier
"two steps" ablation (t → 0 → clean) was not this chain. "Training-free knobs are exhausted" is falsified if chain ×2
− whole ×2 ≥ +0.015 dB at σ 50, pooled over the 110 images, with sign-flip p < 0.01.
Noise: DMID's protocol with the seeds of `diffusion-tweedie-benchmark`. Kaggle: GPU T4 ×2, internet on.

In [ ]:
import io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

SIGMAS = [15, 25, 50]
SETS = ["McMaster", "Kodak24", "CBSD68"]                  # order = noise-seed order of diffusion-tweedie-benchmark
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}


def download(url, path, size=None):
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": ("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", 367_018_356),
}
with ThreadPoolExecutor(2) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TEST = {}
with zipfile.ZipFile(P["datasets"]) as z:
    for ds in SETS:
        names = sorted([n for n in z.namelist() if n.startswith(f"Datasets/Gaussion/{ds}/") and not n.endswith("/")], key=natkey)
        TEST[ds] = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))) for n in names]
print({ds: len(v) for ds, v in TEST.items()}, f"setup {(time.time() - T_START) / 60:.1f} min")


def noisy_of(ds, idx, sigma):
    seed = 1_000_000 * (SETS.index(ds) + 1) + 1000 * sigma + idx
    x = torch.from_numpy(TEST[ds][idx][1]).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


def psnr(out, clean_u8):
    a = (out[0].clamp(0, 1) * 255).round().double()
    b = torch.from_numpy(clean_u8).permute(2, 0, 1).to(a.device).double()
    return 10 * math.log10(255 ** 2 / (a - b).pow(2).mean().item())

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def sdpa_attention(self, qkv):
    """ADM's legacy attention through SDPA with 4-D (batch, heads, tokens, channels) inputs, so the fused kernels apply."""
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.reshape(bs, self.n_heads, ch, length).transpose(2, 3).contiguous()
               for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(2, 3).reshape(bs, -1, length)


gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)


def make_adm(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                     use_scale_shift_norm=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False).convert_to_fp16()
    return m


CTX = [dict(dev=d, adm=make_adm(d)) for d in DEVICES]
ADM_STATE.clear()
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()
ABAR32 = torch.cumprod(1 - torch.from_numpy(np.linspace(1e-4, 0.02, 1000)).float(), 0)
DMID_N = {15: 33, 25: 57, 50: 115}   # DMID main_for_gaussian.py: start at step N − 1


def matched(sigma01):
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


@torch.no_grad()
def tweedie(model, y, sigma01):
    t, ab = matched(sigma01)
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = math.sqrt(ab) * (2 * yp - 1)
    eps = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3].float()
    return (((x - math.sqrt(1 - ab) * eps) / math.sqrt(ab) + 1) / 2)[..., :h, :w]


def tiled(fn, y, tile, overlap):
    _, _, H, W = y.shape
    stride = tile - overlap
    cover = lambda n: max(tile, tile + math.ceil(max(n - tile, 0) / stride) * stride)
    yp = F.pad(y, (0, cover(W) - W, 0, cover(H) - H), mode="reflect")
    r = torch.arange(tile, dtype=torch.float32, device=y.device)
    ramp = torch.clamp(torch.minimum(r + 1, tile - r) / overlap, max=1)
    w2 = ramp[:, None] * ramp[None, :]
    acc, wsum = torch.zeros_like(yp), torch.zeros_like(yp[:, :1])
    for i in range(0, yp.shape[-2] - tile + 1, stride):
        for j in range(0, yp.shape[-1] - tile + 1, stride):
            acc[..., i:i + tile, j:j + tile] += fn(yp[..., i:i + tile, j:j + tile]) * w2
            wsum[..., i:i + tile, j:j + tile] += w2
    return (acc / wsum)[..., :H, :W]


@torch.no_grad()
def dmid_chain(model, y, sigma):
    """DMID's main_for_gaussian sample_image with S_t = 2 at η = 0: x = sqrt(abar_{N-1}) (2y - 1), seq = range(0, N, N // 2)[:-1]
    + [N - 1], deterministic DDIM updates (three evaluations); returns the final x0 in [0, 1] units, unclamped."""
    n = DMID_N[sigma]
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    seq = list(range(0, n, n // 2))[:-1] + [n - 1]
    seq_next = [-1] + seq[:-1]
    x = math.sqrt(float(ABAR32[n - 1])) * (2 * yp - 1)
    for i, j in zip(reversed(seq), reversed(seq_next)):
        at, an = float(ABAR32[i]), (float(ABAR32[j]) if j >= 0 else 1.0)
        eps = model(x, torch.full((x.shape[0],), float(i), device=y.device))[:, :3].float()
        x0 = (x - math.sqrt(1 - at) * eps) / math.sqrt(at)
        x = math.sqrt(an) * x0 + math.sqrt(1 - an) * eps
    return ((x + 1) / 2)[..., :h, :w]

In [ ]:
def run_job(ctx, job):
    ds, idx, sigma = job
    name, clean = TEST[ds][idx]
    y = noisy_of(ds, idx, sigma)[None].to(ctx["dev"])
    one = lambda z: tweedie(ctx["adm"], z, sigma / 255)
    arms = {"whole": one, "tiles-384": lambda z: tiled(one, z, 384, 96), "tiles-512": lambda z: tiled(one, z, 512, 128),
            "DMID chain η=0": lambda z: dmid_chain(ctx["adm"], z, sigma)}
    rows = []
    for arm, fn in arms.items():
        a = fn(y)
        b = fn(y.flip(-1)).flip(-1)
        rows += [dict(dataset=ds, image=name, idx=idx, sigma=sigma, method=f"{arm} ×1", psnr=psnr(a, clean)),
                 dict(dataset=ds, image=name, idx=idx, sigma=sigma, method=f"{arm} ×2", psnr=psnr((a + b) / 2, clean))]
    return rows


def worker_pool(jobs, fn):
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while True:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))
                torch.cuda.empty_cache()

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    while any(th.is_alive() for th in threads):
        time.sleep(10)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
jobs = [(ds, i, s) for ds in SETS for s in SIGMAS for i in range(len(TEST[ds]))]
res, fail = worker_pool(jobs, run_job)
SCORES = pd.DataFrame([r for rows in res for r in rows])
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
print(f"{len(res)}/{len(jobs)} jobs, {len(fail)} failed, {(time.time() - t0) / 60:.1f} min")

In [ ]:
def sign_flip_p(d, n_flip=200_000, seed=0):
    rng = np.random.default_rng(seed)
    signs = rng.choice([-1.0, 1.0], size=(n_flip, len(d)))
    return float((np.abs((signs * d).mean(1)) >= abs(d.mean()) - 1e-12).mean())


print(SCORES.groupby(["dataset", "method", "sigma"]).psnr.mean().unstack().round(3).to_string())
rows = []
for (ds, s), g in SCORES.groupby(["dataset", "sigma"]):
    w = g.pivot_table(index="idx", columns="method", values="psnr")
    for a, b in [("tiles-384 ×2", "whole ×2"), ("tiles-512 ×2", "whole ×2"), ("DMID chain η=0 ×2", "whole ×2")]:
        d = (w[a] - w[b]).to_numpy()
        rows.append(dict(dataset=ds, sigma=s, contrast=f"{a} − {b}", mean=d.mean(), better=int((d > 0).sum()), n=len(d), p=sign_flip_p(d)))
T = pd.DataFrame(rows)
rule = T[T.contrast == "tiles-384 ×2 − whole ×2"].sort_values("p").copy()
rule["p_holm"] = np.minimum(1.0, np.maximum.accumulate(rule.p.to_numpy() * (len(rule) - np.arange(len(rule)))))
T = T.merge(rule[["dataset", "sigma", "contrast", "p_holm"]], on=["dataset", "sigma", "contrast"], how="left")
print(T.round(4).to_string(index=False))
chain = SCORES.pivot_table(index=["dataset", "idx", "sigma"], columns="method", values="psnr")
P1C = {}
for s in SIGMAS:
    d = (chain["DMID chain η=0 ×2"] - chain["whole ×2"]).xs(s, level="sigma").to_numpy()
    P1C[s] = dict(mean=float(d.mean()), better=int((d > 0).sum()), n=len(d), p=sign_flip_p(d))
print("P1c, DMID chain η=0 ×2 − whole ×2, pooled:", P1C)
print("P1c verdict: 'knobs exhausted' falsified:", P1C[50]["mean"] >= 0.015 and P1C[50]["p"] < 0.01)
T.to_csv(f"{OUT}/paired.csv", index=False)
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes=(time.time() - T_START) / 60, paired=T.to_dict("records"), p1c=P1C, failed=[j for j, _ in fail]),
              f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))